The [TensorFlow Embedding Projector](https://projector.tensorflow.org/) places
high-dimensional word vectors in a three-dimensional map where distance approximates
semantic similarity, and lets you pick a word to see its nearest neighbors. In this
assignment you build the same thing in PyTorch: you train word embeddings with
`torch.nn.Embedding`, project them to three dimensions, draw an interactive scatter, and
query the neighborhood of any token.

You will complete the parts marked with `TODO(you)`. Each raises `NotImplementedError`
until you implement it.

In [3]:
import re
from collections import Counter
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

## Corpus and vocabulary

Word embeddings are learned from co-occurrence in text. Load a compact corpus, keep the
most frequent words as the vocabulary, and turn the text into a stream of integer ids.

In [4]:
from datasets import load_dataset

raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
text = " ".join(raw["text"]).lower()
tokens = re.findall(r"[a-z]+", text)[:300_000]
counts = Counter(tokens)

V = 8000
# TODO(you): build `vocab` (the V most common words), `word2idx`, `idx2word`,
# and `corpus` (the token stream mapped to ids, dropping out-of-vocabulary words).
vocab = [word for word, _ in counts.most_common(V)]
word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for word, i in word2idx.items()}

corpus = []
for word in tokens:
    if word in word2idx:
        corpus.append(word2idx[word])

print("vocab size:", len(vocab))
print("corpus length:", len(corpus))
print("first words:", vocab[:10])

/home/nick/NicksWorkspace/eng-ai-agents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating validation split: 100%|██████████| 3760/3760 [00:00<00:00, 617099.04 examples/s]


vocab size: 8000
corpus length: 276042
first words: ['the', 'of', 'and', 'in', 'to', 'a', 'was', 's', 'that', 'as']


## Word2vec embeddings with softmax and cross-entropy

A word2vec model learns word embeddings by predicting context words. This is the skip-gram
architecture of word2vec: the center word predicts its context. The center word's embedding is
scored against every word in the vocabulary, a softmax turns those scores into a probability
distribution over possible context words, and the cross-entropy loss pushes up the probability
of the true context word:

$$p(o \mid c) = \frac{\exp(\mathbf{c}\cdot\mathbf{v}_o)}{\sum_{w}\exp(\mathbf{c}\cdot\mathbf{v}_w)},
\qquad L = -\log p(o \mid c).$$

The learned center embedding table is the word-vector matrix you will project.

In [5]:
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.center = nn.Embedding(vocab_size, dim)   # word vectors
        self.output = nn.Linear(dim, vocab_size)      # score every word as a possible context
        nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)

    def forward(self, center_ids):
        # TODO(you): embed the center ids and return the (B, V) scores over the whole
        # vocabulary (one score per possible context word). Cross-entropy + softmax are applied
        # by the loss in the training loop, so return the raw scores (logits), not probabilities.
        x = self.center(center_ids)
        logits = self.output(x)
        return logits

In [6]:
# Build (center, context) pairs from a sliding window
window = 3
pairs = []
for i, wc in enumerate(corpus):
    for j in range(max(0, i - window), min(len(corpus), i + window + 1)):
        if j != i:
            pairs.append((wc, corpus[j]))
pairs = np.array(pairs, dtype=np.int64)

dim, B, epochs = 64, 1024, 3
model = Word2Vec(V, dim)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()

# TODO(you): write the training loop. For each mini-batch, get the (B, V) logits from the center
# ids with model(...), compute the cross-entropy loss against the true context ids with loss_fn,
# backprop, and step the optimizer. Track the per-epoch loss. After training, set
# `emb = model.center.weight.detach().cpu().numpy()`.
epoch_losses = []

for epoch in range(epochs):
    order = np.random.permutation(len(pairs))
    shuffled = pairs[order]
    total_loss = 0.0
    total_examples = 0

    for start in range(0, len(shuffled), B):
        batch = shuffled[start:start + B]
        center_ids = torch.tensor(batch[:, 0], dtype=torch.long)
        context_ids = torch.tensor(batch[:, 1], dtype=torch.long)

        logits = model(center_ids)
        loss = loss_fn(logits, context_ids)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item() * len(batch)
        total_examples += len(batch)

    avg_loss = total_loss / total_examples
    epoch_losses.append(avg_loss)
    print(f"epoch {epoch + 1}/{epochs} loss {avg_loss:.4f}")

emb = model.center.weight.detach().cpu().numpy()

epoch 1/3 loss 6.9860
epoch 2/3 loss 6.7126
epoch 3/3 loss 6.5749


## Projecting the embeddings to three dimensions

The embedding matrix lives in $d=64$ dimensions. To see it, project a few thousand of the most
frequent words down to three dimensions. Principal component analysis is linear and fast; UMAP is
nonlinear and tends to separate clusters more sharply. The interactive scatter lets you rotate the
cloud and hover to read each word.

In [8]:
from sklearn.decomposition import PCA

N = 1500
plot_words = vocab[:N]
X = emb[:N]

# TODO(you): compute `pca3` (N x 3) with PCA, and `umap3` with UMAP (guard UMAP in a
# try/except so a missing umap-learn does not crash the notebook).
pca3 = PCA(n_components=3).fit_transform(X)

try:
    import umap
    umap3 = umap.UMAP(n_components=3, random_state=0).fit_transform(X)
    print("UMAP projection computed.")
except Exception as e:
    umap3 = None
    print("UMAP not available:", e)

print("PCA shape:", pca3.shape)
if umap3 is not None:
    print("UMAP shape:", umap3.shape)

/home/nick/NicksWorkspace/eng-ai-agents/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP projection computed.
PCA shape: (1500, 3)
UMAP shape: (1500, 3)


In [13]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

# TODO(you): write `plot_embeddings(coords, words, query=None, neighbor_set=None)` that
# draws a plotly Scatter3d: hover text = the word; color/size the `query` and any words in
# `neighbor_set` distinctly. Return the figure (end the cell with the figure object).
def plot_embeddings(coords, words, query=None, neighbor_set=None):
    if neighbor_set is None:
        neighbor_set = set()

    colors = []
    sizes = []
    for word in words:
        if query is not None and word == query:
            colors.append("red")
            sizes.append(8)
        elif word in neighbor_set:
            colors.append("orange")
            sizes.append(5)
        else:
            colors.append("steelblue")
            sizes.append(3)

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=coords[:, 0],
                y=coords[:, 1],
                z=coords[:, 2],
                mode="markers",
                text=words,
                hovertemplate="%{text}<extra></extra>",
                marker=dict(size=sizes, color=colors, opacity=0.8),
            )
        ]
    )
    fig.update_layout(
        title="Embedding projector (PCA)",
        scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
        width=900,
        height=700,
    )
    return fig


plot_embeddings(pca3, plot_words)

## Querying a token's neighborhood

The projector's key feature is the neighborhood query: pick a word and see its closest
words. Closeness is measured by cosine similarity in the full embedding space (not in the
3D projection). The query below returns the top-k neighbors and highlights them in the
scatter.

In [14]:
def neighbors(word, k=10):
    # TODO(you): return the k nearest words to `word` by cosine similarity over `emb`,
    # as a list of (word, score) sorted by descending score, excluding `word` itself.
    if word not in word2idx:
        raise ValueError(f"{word!r} is not in the vocabulary")

    idx = word2idx[word]
    query_vec = emb[idx]

    query_norm = np.linalg.norm(query_vec)
    emb_norms = np.linalg.norm(emb, axis=1)
    sims = emb @ query_vec / (emb_norms * query_norm + 1e-12)

    order = np.argsort(-sims)
    result = []
    for j in order:
        if j == idx:
            continue
        result.append((idx2word[j], float(sims[j])))
        if len(result) == k:
            break
    return result

for w, s in neighbors("government", 10):
    print(f"{w:15s} {s:.3f}")

municipal       0.825
federal         0.823
troops          0.822
revolutionary   0.821
commonwealth    0.820
pakistani       0.813
subcontinent    0.811
courts          0.810
fledgling       0.808
invasion        0.804


In [15]:
# TODO(you): pick a query word, get its neighbors with neighbors(query, 10), and re-draw the
# projector with plot_embeddings(...) highlighting the query and its neighbors. A word only
# appears in the plot if it is among the N most frequent words used for pca3.
query = "government"
neighbor_list = neighbors(query, 10)
neighbor_set = {word for word, _ in neighbor_list if word in plot_words}

print("query:", query)
print("neighbors:")
for word, score in neighbor_list:
    print(f"{word:15s} {score:.3f}")

plot_embeddings(pca3, plot_words, query=query, neighbor_set=neighbor_set)

query: government
neighbors:
municipal       0.825
federal         0.823
troops          0.822
revolutionary   0.821
commonwealth    0.820
pakistani       0.813
subcontinent    0.811
courts          0.810
fledgling       0.808
invasion        0.804


## Exploration

Answer in the cells you add below.

1. Query several words of your choice (a few nouns, a verb, a function word). Which return clean
   semantic neighbors and which do not? Why might rare words give noisier neighbors?
2. Plot the clusters. Draw the projected embeddings (the UMAP layout separates clusters most
   clearly) and describe the groupings you see: do related words land near each other? Name a few
   clusters you can identify.

In [18]:
query_words = ["government", "city", "music", "run", "the", "people"]

for word in query_words:
    print(f"\nNeighbors for: {word}")
    for neighbor_word, score in neighbors(word, 10):
        print(f"  {neighbor_word:15s} {score:.3f}")



Neighbors for: government
  municipal       0.825
  federal         0.823
  troops          0.822
  revolutionary   0.821
  commonwealth    0.820
  pakistani       0.813
  subcontinent    0.811
  courts          0.810
  fledgling       0.808
  invasion        0.804

Neighbors for: city
  council         0.828
  downtown        0.818
  sarnia          0.794
  legislative     0.771
  michigan        0.770
  economy         0.767
  mayor           0.753
  municipal       0.753
  library         0.752
  census          0.750

Neighbors for: music
  accompanying    0.869
  pop             0.805
  concept         0.793
  susan           0.783
  video           0.757
  sony            0.755
  glenn           0.752
  ambient         0.751
  videos          0.749
  producer        0.735

Neighbors for: run
  cheltenham      0.808
  equalised       0.777
  fleetwood       0.774
  struggling      0.767
  drew            0.764
  defeat          0.746
  away            0.742
  episodes        0.74

In [19]:
if umap3 is not None:
    fig = plot_embeddings(umap3, plot_words)
    fig.update_layout(title="Embedding projector (UMAP)")
    fig
else:
    fig = plot_embeddings(pca3, plot_words)
    fig.update_layout(title="Embedding projector (PCA fallback)")
    fig


## Exploration Answers

### 1. Query several words of your choice. Which return clean semantic neighbors and which do not? Why might rare words give noisier neighbors?

Some words produced cleaner semantic neighborhoods than others. `government` gave the clearest results. Its nearest words included `municipal`, `federal`, `courts`, `commonwealth`, and `revolutionary`, which all fit a political or institutional theme. `city` also returned fairly good neighbors such as `council`, `downtown`, `mayor`, `municipal`, `library`, and `census`, which are all connected to cities or local government. `music` was somewhat mixed: words like `pop`, `ambient`, `producer`, and `video` make sense, but names like `susan` and `glenn` are less clean and probably reflect specific article contexts in the training corpus.

`run` was much noisier. Its neighbors included words like `cheltenham`, `fleetwood`, `drew`, and `episodes`, which do not form one clear semantic group. That makes sense because `run` is used in many different ways, so its contexts are more spread out. The function word `the` gave the weakest result. Its neighbors did not form a meaningful semantic set at all, which is expected because function words are mostly grammatical and appear in almost every topic. `people` was in the middle: it returned some sensible neighbors like `individuals`, `businesses`, and `types`, but also several broader or more context-dependent words.

Rare words often give noisier neighbors because the model sees them fewer times during training. With fewer updates, their embeddings do not settle into as stable a position in the vector space. Common content words usually learn cleaner neighborhoods because the model has many chances to observe their contexts. Very common function words can still look weak semantically, because their role is grammatical rather than topical.

### 2. Plot the clusters. Draw the projected embeddings and describe the groupings you see: do related words land near each other? Name a few clusters you can identify.

In the projected embedding plots, related words do tend to land near each other, although the separation is not perfect. The UMAP layout is easier to read than PCA because it separates local neighborhoods more clearly, while PCA gives a flatter overall view. One cluster that stands out is the government or political group, including words like `government`, `municipal`, `federal`, and `courts`. Another visible group is built around city or civic terms such as `city`, `council`, `mayor`, `downtown`, and `census`. There also appears to be a music-related region with words like `music`, `pop`, `ambient`, `producer`, and `video`.

Function words such as `the` do not seem to form a strong semantic cluster, which matches the weaker neighbor results. More generally, the plot suggests that the model learned real structure from word co-occurrence: topical words often gather into meaningful neighborhoods, while broader or highly flexible words are more scattered. So the embedding space is not perfect, but it does capture a noticeable amount of semantic organization.
